# OSCILLATIONS AND ACCURACY OF LINEARIZATION

In [1]:
import matplotlib.pyplot as plt  # type: ignore
import numpy as np  # type: ignore
import os
import pandas as pd  # type: ignore
from typing import Optional, cast, List, Union

import src.constants as cn
from src.score import Score
from src.system_discovery import SystemDiscovery
from src.piecewise_system_discovery import PiecewiseSystemDiscovery
from src.model import Model
from src.timecourse import Timecourse
from src.timecourse_iterator import TimecourseIterator
from src.system_discovery_changepoint_detector import SystemDiscoveryChangepointDetector

# Selecting Oscillating Models and Species

In [2]:
# Select oscillating and non-oscillating species
path = os.path.join(cn.DATA_DIR, f"find_oscillating_models.csv")
df = pd.read_csv(path)
df["key"] = [m + "_" + str(s) for m, s in zip(df[cn.COL_SYSTEM_ID], df[cn.COL_SPECIES_NAME])]
sel = df[cn.COL_FREQUENCIES] == "[]"
OSCILLATING_DF = df[~sel]
NONOSCILLATION_DF = df [sel]
NONOSCILLATION_DF

,system_id,species_name,frequencies,endtime,key
37,BIOMD0000000042,GLC,[],300.0,BIOMD0000000042_GLC
39,BIOMD0000000042,FBP,[],300.0,BIOMD0000000042_FBP
40,BIOMD0000000042,GAP,[],300.0,BIOMD0000000042_GAP
41,BIOMD0000000042,NAD,[],300.0,BIOMD0000000042_NAD
42,BIOMD0000000042,NADH,[],300.0,BIOMD0000000042_NADH
...,...,...,...,...,...
1107209,BIOMD0000001059,sursmac,[],7000.0,BIOMD0000001059_sursmac
1107210,BIOMD0000001059,smacmit,[],7000.0,BIOMD0000001059_smacmit
1107211,BIOMD0000001059,smac,[],7000.0,BIOMD0000001059_smac
1107212,BIOMD0000001060,x1,[],40.0,BIOMD0000001060_x1


In [3]:
OSCILLATING_DF

,system_id,species_name,frequencies,endtime,key
0,BIOMD0000000005,C2,[0.02997],100.0,BIOMD0000000005_C2
1,BIOMD0000000005,CP,[0.02997],100.0,BIOMD0000000005_CP
2,BIOMD0000000005,M,"[0.02997, 0.05994, 0.10989, 0.13986]",100.0,BIOMD0000000005_M
3,BIOMD0000000005,pM,[0.02997],100.0,BIOMD0000000005_pM
4,BIOMD0000000005,Y,[0.02997],100.0,BIOMD0000000005_Y
...,...,...,...,...,...
1107197,BIOMD0000001058,Polo,[0.00999],200.0,BIOMD0000001058_Polo
1107198,BIOMD0000001058,Sic1t,"[0.00999, 0.01998]",200.0,BIOMD0000001058_Sic1t
1107199,BIOMD0000001058,SBF,[0.00999],200.0,BIOMD0000001058_SBF
1107200,BIOMD0000001058,Cdh1,"[0.00999, 0.01998]",200.0,BIOMD0000001058_Cdh1


# Individual models

In [ ]:
# Oscillating models
species_names = ["MBF", "ClbSt"]
#species_names = None
for model_num in [1058]:
    model = Model.makeBiomodel(model_num=model_num)
    timecourse = Timecourse(model, num_point =1000)
    timecourse.plot(is_scatter=True, species_names=species_names)
    df = timecourse.timecourse_df
    try:
        psd = PiecewiseSystemDiscovery(df, max_changepoint=80, min_segment_length=10,
                                        max_fractional_reduction=0.01, model_name=str(model_num))
        psd.fit()
        result = psd.plotPiecewise(legend=False, num_true_point=60, species_names=species_names)
    except Exception as e:
        print(f"Could not process {model_num}: {e}")